In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
!pip install -q datasets transformers pandas scikit-learn

In [2]:
import json
import re
import os
import pandas as pd
from datasets import load_dataset

print("All libraries imported successfully!")
print(f"Pandas version    : {pd.__version__}")

All libraries imported successfully!
Pandas version    : 2.3.3


In [3]:
# This downloads IMDb directly from Hugging Face (~80MB)
# Internet must be ON in Kaggle settings for this to work

print("Downloading IMDb dataset from Hugging Face...")
dataset = load_dataset("imdb")

# Convert splits to DataFrames
train_df = pd.DataFrame(dataset["train"])
test_df  = pd.DataFrame(dataset["test"])

print("\n✅ Dataset loaded successfully!")
print(f"Train size : {len(train_df):,} rows")
print(f"Test size  : {len(test_df):,} rows")
print(f"Columns    : {list(train_df.columns)}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


✅ Dataset loaded successfully!
Train size : 25,000 rows
Test size  : 25,000 rows
Columns    : ['text', 'label']


In [4]:
# ── Size and structure ────────────────────────────────────────
print("=" * 50)
print("RAW DATA INSPECTION")
print("=" * 50)

print(f"\nShape of train set : {train_df.shape}")
print(f"Shape of test set  : {test_df.shape}")

# ── Class distribution ────────────────────────────────────────
print("\nLabel distribution in TRAIN set:")
print(train_df["label"].value_counts())
print(f"\nClass balance: {train_df['label'].value_counts(normalize=True).mul(100).round(1).to_dict()}")

# ── Missing values ────────────────────────────────────────────
print("\nMissing values in train:")
print(train_df.isnull().sum())

print("\nMissing values in test:")
print(test_df.isnull().sum())

# ── Sample raw review ─────────────────────────────────────────
print("\nSample RAW review (first 400 chars):")
print("-" * 40)
print(train_df["text"].iloc[0][:400])
print("-" * 40)
print(f"\nLabel : {train_df['label'].iloc[0]} ({'positive' if train_df['label'].iloc[0]==1 else 'negative'})")

RAW DATA INSPECTION

Shape of train set : (25000, 2)
Shape of test set  : (25000, 2)

Label distribution in TRAIN set:
label
0    12500
1    12500
Name: count, dtype: int64

Class balance: {0: 50.0, 1: 50.0}

Missing values in train:
text     0
label    0
dtype: int64

Missing values in test:
text     0
label    0
dtype: int64

Sample RAW review (first 400 chars):
----------------------------------------
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student name
----------------------------------------

Label : 0 (negative)


In [5]:
def clean_text(text):
    """
    Cleaning steps applied (justify each in your report):

    1. Lowercase      : Sentiment doesn't depend on case
    2. Remove HTML    : IMDb reviews contain raw <br /> tags
    3. Remove punctuation : Reduces noise for tokenizer
    4. Collapse whitespace: Normalises spacing after removals
    """
    text = text.lower()                          # 1. lowercase
    text = re.sub(r"<br\s*/?>", " ", text)       # 2. remove HTML <br> tags
    text = re.sub(r"[^a-z0-9\s]", "", text)      # 3. remove punctuation/symbols
    text = re.sub(r"\s+", " ", text).strip()     # 4. collapse extra whitespace
    return text


# Apply to both splits
print("Cleaning train set...")
train_df["text"] = train_df["text"].apply(clean_text)

print("Cleaning test set...")
test_df["text"] = test_df["text"].apply(clean_text)

# Show before vs after comparison
print("\n✅ Cleaning complete!")
print("\nSample CLEANED review (first 400 chars):")
print("-" * 40)
print(train_df["text"].iloc[0][:400])
print("-" * 40)

Cleaning train set...
Cleaning test set...

✅ Cleaning complete!

Sample CLEANED review (first 400 chars):
----------------------------------------
i rented i am curiousyellow from my video store because of all the controversy that surrounded it when it was first released in 1967 i also heard that at first it was seized by us customs if it ever tried to enter this country therefore being a fan of films considered controversial i really had to see this for myself the plot is centered around a young swedish drama student named lena who wants to
----------------------------------------


In [6]:
# ── Train set ─────────────────────────────────────────────────
before_train = len(train_df)
train_df.drop_duplicates(subset="text", inplace=True)
train_df.dropna(subset=["text", "label"], inplace=True)
after_train = len(train_df)

# ── Test set ──────────────────────────────────────────────────
before_test = len(test_df)
test_df.drop_duplicates(subset="text", inplace=True)
test_df.dropna(subset=["text", "label"], inplace=True)
after_test = len(test_df)

print("=" * 40)
print("CLEANING SUMMARY")
print("=" * 40)
print(f"Train : {before_train:,} → {after_train:,} rows ({before_train - after_train} removed)")
print(f"Test  : {before_test:,}  → {after_test:,}  rows ({before_test - after_test} removed)")

print("\nLabel distribution AFTER cleaning (train):")
print(train_df["label"].value_counts())

CLEANING SUMMARY
Train : 25,000 → 24,902 rows (98 removed)
Test  : 25,000  → 24,798  rows (202 removed)

Label distribution AFTER cleaning (train):
label
1    12471
0    12431
Name: count, dtype: int64


In [7]:
import json

# Define label mappings
id2label = {"0": "negative", "1": "positive"}
label2id = {"negative": 0,   "positive": 1}

mapping = {
    "id2label": id2label,
    "label2id": label2id
}

# Save to Kaggle working directory
os.makedirs("/kaggle/working/data", exist_ok=True)

with open("/kaggle/working/data/id2label.json", "w") as f:
    json.dump(mapping, f, indent=2)

print("✅ Saved: /kaggle/working/data/id2label.json")
print("\nContents:")
print(json.dumps(mapping, indent=2))

✅ Saved: /kaggle/working/data/id2label.json

Contents:
{
  "id2label": {
    "0": "negative",
    "1": "positive"
  },
  "label2id": {
    "negative": 0,
    "positive": 1
  }
}


In [8]:
# Save cleaned data to Kaggle working directory
train_df.to_csv("/kaggle/working/data/train_clean.csv", index=False)
test_df.to_csv("/kaggle/working/data/test_clean.csv",   index=False)

print("✅ Saved: /kaggle/working/data/train_clean.csv")
print("✅ Saved: /kaggle/working/data/test_clean.csv")

# Verify file sizes
import os
train_size = os.path.getsize("/kaggle/working/data/train_clean.csv") / (1024*1024)
test_size  = os.path.getsize("/kaggle/working/data/test_clean.csv")  / (1024*1024)

print(f"\nFile sizes:")
print(f"  train_clean.csv : {train_size:.1f} MB")
print(f"  test_clean.csv  : {test_size:.1f} MB")
print("\n⚠️  Note: These CSVs stay on Kaggle only.")
print("Only id2label.json gets committed to GitHub.")

✅ Saved: /kaggle/working/data/train_clean.csv
✅ Saved: /kaggle/working/data/test_clean.csv

File sizes:
  train_clean.csv : 30.0 MB
  test_clean.csv  : 29.2 MB

⚠️  Note: These CSVs stay on Kaggle only.
Only id2label.json gets committed to GitHub.


In [9]:
!pip install -q transformers datasets wandb scikit-learn accelerate huggingface_hub
print('✅ Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [10]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')

import wandb
from huggingface_hub import login

login(token=HF_TOKEN)
wandb.login()
print('✅ Logged in to W&B and Hugging Face.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: pinakijayakar01 (pinakijayakar01-iitj) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged in to W&B and Hugging Face.


In [11]:
# ── Shared setup: load cleaned data + id2label from Task 2 ──────────────────
import json
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

# Load CSVs saved by Task 2
train_df = pd.read_csv('/kaggle/working/data/train_clean.csv')
test_df  = pd.read_csv('/kaggle/working/data/test_clean.csv')

# Subsample to stay within Kaggle free GPU limits
TRAIN_SIZE = 10_000
EVAL_SIZE  =  2_000

train_df = train_df.sample(n=TRAIN_SIZE, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(n=EVAL_SIZE,  random_state=42).reset_index(drop=True)

print(f'Train size : {len(train_df):,}')
print(f'Eval size  : {len(test_df):,}')
print(f'Label distribution (train):\n{train_df["label"].value_counts()}')

# Load id2label from Task 2 (string keys -> convert to int)
with open('/kaggle/working/data/id2label.json') as f:
    mapping = json.load(f)

id2label = {int(k): v for k, v in mapping['id2label'].items()}
label2id = {v: int(k) for k, v in mapping['id2label'].items()}

print('\nid2label:', id2label)
print('label2id:', label2id)

Train size : 10,000
Eval size  : 2,000
Label distribution (train):
label
0    5012
1    4988
Name: count, dtype: int64

id2label: {0: 'negative', 1: 'positive'}
label2id: {'negative': 0, 'positive': 1}


In [12]:
# ── Tokenize once — both versions share the same tokenized datasets ──────────
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

train_hf = Dataset.from_pandas(train_df[['text', 'label']])
eval_hf  = Dataset.from_pandas(test_df[['text', 'label']])

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_hf = train_hf.map(tokenize, batched=True)
eval_hf  = eval_hf.map(tokenize,  batched=True)

train_hf = train_hf.rename_column('label', 'labels')
eval_hf  = eval_hf.rename_column('label',  'labels')

train_hf.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_hf.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Tokenization complete.')
print(f'   Train: {train_hf}')
print(f'   Eval : {eval_hf}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Tokenization complete.
   Train: Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 10000
})
   Eval : Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2000
})


In [13]:
# ── Helper: run one experiment version ───────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

def run_experiment(version, learning_rate, epochs=3, batch_size=16,
                   warmup_steps=200, weight_decay=0.01):
    """
    Fine-tunes DistilBERT for one experiment version.
    Returns (trainer, results_dict) so the best model can be pushed in Task 5.
    """
    print(f'\n{"="*55}')
    print(f'  Starting {version}  |  LR={learning_rate}  |  Epochs={epochs}')
    print(f'{"="*55}\n')

    # Fresh model for each run (no weight leakage between versions)
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    wandb.init(
        project='mlops-assignment3',
        name=f'run-{version}',
        config={
            'model_name':    MODEL_NAME,
            'epochs':        epochs,
            'batch_size':    batch_size,
            'learning_rate': learning_rate,
            'warmup_steps':  warmup_steps,
            'weight_decay':  weight_decay,
            'version':       version,
            'platform':      'Kaggle',
            'dataset':       'IMDb',
        },
    )

    args = TrainingArguments(
        output_dir                  = f'/kaggle/working/results-{version}',
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = learning_rate,
        warmup_steps                = warmup_steps,
        weight_decay                = weight_decay,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        report_to                   = 'wandb',
        run_name                    = f'run-{version}',
        logging_steps               = 50,
        fp16                        = True,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_hf,
        eval_dataset    = eval_hf,
        compute_metrics = compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()

    print(f'\n✅ {version} results: {results}')

    wandb.run.summary['final_accuracy'] = results['eval_accuracy']
    wandb.run.summary['final_f1']       = results['eval_f1']
    wandb.run.summary['final_loss']     = results['eval_loss']
    wandb.finish()

    return trainer, results

print('✅ run_experiment() defined.')

✅ run_experiment() defined.


In [14]:
trainer_v1, results_v1 = run_experiment(
    version       = 'v1',
    learning_rate = 3e-5,
)


  Starting v1  |  LR=3e-05  |  Epochs=3



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



✅ v1 results: {'eval_loss': 0.6712769269943237, 'eval_accuracy': 0.905, 'eval_f1': 0.9050011400410415, 'eval_runtime': 8.5626, 'eval_samples_per_second': 233.573, 'eval_steps_per_second': 7.358, 'epoch': 3.0}


eval/accuracy,▁▄██
eval/f1,▁▄██
eval/loss,▂▁██
eval/runtime,█▂▂▁
eval/samples_per_second,▁▇▇█
eval/steps_per_second,▁▇▇█
train/epoch,▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
train/global_step,▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
train/grad_norm,▂▅▆▅▄▄▆▄▃▆▃█▁▁▂▆▅▅
train/learning_rate,▂▄▆██▇▇▆▆▅▅▄▄▃▃▂▂▁
+1,...


In [15]:
trainer_v2, results_v2 = run_experiment(
    version       = 'v2',
    learning_rate = 5e-5,
)


  Starting v2  |  LR=5e-05  |  Epochs=3



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.598689,0.799014,0.837000,0.834115
2,0.372430,0.499563,0.909000,0.908996
3,0.152648,0.746673,0.911000,0.911000


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



✅ v2 results: {'eval_loss': 0.7466728687286377, 'eval_accuracy': 0.911, 'eval_f1': 0.911, 'eval_runtime': 8.5892, 'eval_samples_per_second': 232.852, 'eval_steps_per_second': 7.335, 'epoch': 3.0}


eval/accuracy,▁███
eval/f1,▁███
eval/loss,█▁▇▇
eval/runtime,█▅▃▁
eval/samples_per_second,▁▄▆█
eval/steps_per_second,▁▄▆█
train/epoch,▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
train/global_step,▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
train/grad_norm,▂▄█▄▅▃▇▃▅▆▃▅▁▂▃▆▄▁
train/learning_rate,▂▄▆██▇▇▆▆▅▅▄▄▃▃▂▂▁
+1,...


In [16]:
# ── Compare v1 vs v2 ─────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  EXPERIMENT COMPARISON')
print('='*55)
print(f'{"Metric":<20} {"v1 (LR=3e-5)":>15} {"v2 (LR=5e-5)":>15}')
print('-'*55)
print(f'{"Accuracy":<20} {results_v1["eval_accuracy"]:>15.4f} {results_v2["eval_accuracy"]:>15.4f}')
print(f'{"F1 (weighted)":<20} {results_v1["eval_f1"]:>15.4f} {results_v2["eval_f1"]:>15.4f}')
print(f'{"Val Loss":<20} {results_v1["eval_loss"]:>15.4f} {results_v2["eval_loss"]:>15.4f}')
print('='*55)

best_version = 'v1' if results_v1['eval_f1'] >= results_v2['eval_f1'] else 'v2'
best_trainer = trainer_v1 if best_version == 'v1' else trainer_v2
print(f'\n🏆 Best version: {best_version}')


  EXPERIMENT COMPARISON
Metric                  v1 (LR=3e-5)    v2 (LR=5e-5)
-------------------------------------------------------
Accuracy                      0.9050          0.9110
F1 (weighted)                 0.9050          0.9110
Val Loss                      0.6713          0.7467

🏆 Best version: v2


In [17]:
# ── Task 5: Push best model to Hugging Face Hub ───────────────────────────────

HF_REPO_ID   = 'pjayG25AIT2045/distilbert-imdb-mlops'
HF_MODEL_URL = f'https://huggingface.co/{HF_REPO_ID}'

print(f'Pushing best model ({best_version}) to: {HF_MODEL_URL}')

best_trainer.model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print('✅ Push complete.')

# Log HF model URL into W&B run summary
wandb.init(
    project = 'mlops-assignment3',
    name    = f'model-push-{best_version}',
    resume  = 'allow',
)
wandb.run.summary['huggingface_model'] = HF_MODEL_URL
wandb.finish()

print(f'✅ HF model URL logged to W&B: {HF_MODEL_URL}')

Pushing best model (v2) to: https://huggingface.co/pjayG25AIT2045/distilbert-imdb-mlops


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✅ Push complete.


huggingface_model,https://huggingface....


✅ HF model URL logged to W&B: https://huggingface.co/pjayG25AIT2045/distilbert-imdb-mlops
